In [ ]:
from mainnet_launch.constants import *
from mainnet_launch.abis import *

from mainnet_launch.data_fetching.get_events import *
from mainnet_launch.data_fetching.get_state_by_block import *

from fetch_usd_asset_discounts import (
    build_autoUSD_token_backing_calls,
    build_autoUSD_token_safe_price_calls,
    stablecoin_tuples,
)

# some goals

# get token prices from on chain sources


backing_calls = build_autoUSD_token_backing_calls()
safe_price_calls = build_autoUSD_token_safe_price_calls()

blocks = build_blocks_to_use(ETH_CHAIN, 22032640)

df = get_raw_state_by_blocks([*backing_calls, *safe_price_calls], blocks, ETH_CHAIN)

In [ ]:
backing_cols = [col for col in df.columns if col.endswith("_backing")]
df_backing = df[backing_cols].copy()
df_backing.columns = [col.replace("_backing", "") for col in df_backing.columns]

safe_price_cols = [col for col in df.columns if col.endswith("_safe_price")]
df_safe_price = df[safe_price_cols].copy()
df_safe_price.columns = [col.replace("_safe_price", "") for col in df_safe_price.columns]

common_tokens = sorted(set(df_backing.columns) & set(df_safe_price.columns))
df_discount = pd.DataFrame(index=df.index, columns=common_tokens)
for token in common_tokens:
    df_discount[token] = 100 * (1 - df_backing[token] / df_safe_price[token])

Backing Table:
                           DAI  USDe  USDC  USDT  GHO  crvUSD  USDs  FRAX  \
timestamp                                                                   
2025-03-12 23:59:59+00:00  1.0   1.0   1.0   1.0  1.0     1.0   1.0   1.0   
2025-03-13 23:59:59+00:00  1.0   1.0   1.0   1.0  1.0     1.0   1.0   1.0   
2025-03-14 23:59:59+00:00  1.0   1.0   1.0   1.0  1.0     1.0   1.0   1.0   
2025-03-15 23:59:59+00:00  1.0   1.0   1.0   1.0  1.0     1.0   1.0   1.0   
2025-03-16 23:59:59+00:00  1.0   1.0   1.0   1.0  1.0     1.0   1.0   1.0   

                              aUSDT     aUSDC      aGHO     sUSDs     sUSDe  \
timestamp                                                                     
2025-03-12 23:59:59+00:00  1.117750  1.123344  1.002190  1.042824  1.162270   
2025-03-13 23:59:59+00:00  1.117833  1.123428  1.002241  1.043004  1.162426   
2025-03-14 23:59:59+00:00  1.117915  1.123515  1.002298  1.043184  1.162544   
2025-03-15 23:59:59+00:00  1.117999  1.123601  1.0

In [5]:
df.columns

Index(['DAI_backing', 'USDe_backing', 'USDC_backing', 'USDT_backing',
       'GHO_backing', 'crvUSD_backing', 'USDs_backing', 'FRAX_backing',
       'aUSDT_backing', 'aUSDC_backing', 'aGHO_backing', 'sUSDs_backing',
       'sUSDe_backing', 'scrvUSD_backing', 'sDAI_backing', 'sFRAX_backing',
       'DAI_safe_price', 'USDC_safe_price', 'USDT_safe_price',
       'GHO_safe_price', 'USDs_safe_price', 'USDe_safe_price',
       'FRAX_safe_price', 'crvUSD_safe_price', 'scrvUSD_safe_price',
       'sUSDs_safe_price', 'sUSDe_safe_price', 'sFRAX_safe_price',
       'aUSDT_safe_price', 'aUSDC_safe_price', 'aGHO_safe_price'],
      dtype='object')

In [2]:
# solver_root_oracle_contract.all_functions()
# # solver root oracle .getPrice in quote(token, USDC) -> safe price (in 1e6) at the block
# usde = "0x4c9EDD5852cd905f086C759E8383e09bff1E68B3"
# usdc = "0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48"
# solver_root_oracle_contract.functions.getPriceInQuote(usde, usdc).call()
# # this works for the safe price ofthe other tokens
# self_spot_contract = eth_client.eth.contract(SelfSpotEthOracle, abi=SELF_SPOT_ETH_ORACLE_ABI)
# solver_root_oracle_contract = eth_client.eth.contract(SolverRootOracle, abi=SOLVER_ROOT_ORACLE_ABI)
# # getPriceInQuote

# for e in solver_root_oracle_contract.events:
#     print(e)